# 04 — Fairness / Subgroup Performance Check

Healthcare models must be audited for uneven performance or error rates across protected subgroups (race, gender, age) before any real-world use.

In [1]:
import pandas as pd
import joblib
import sys
sys.path.append('../src')
from sklearn.metrics import recall_score, precision_score

In [2]:
df = pd.read_csv('../data/processed/model_ready.csv')
model = joblib.load('../data/processed/readmission_model.pkl')

## Score the full population
(For a real audit, use only the held-out test set with subgroup labels retained.)

In [3]:
X = df.drop(columns=['patient_nbr','readmitted_30d'])
y = df['readmitted_30d']
df['pred_proba'] = model.predict_proba(X)[:,1]
df['pred'] = model.predict(X)

## Recall and precision by subgroup
Look for meaningfully lower recall (missed high-risk patients) in any subgroup — that's the harm to flag.

In [4]:
for group_col in ['race','gender','age']:
    print(f'--- {group_col} ---')
    summary = df.groupby(group_col).apply(
        lambda g: pd.Series({
            'n': len(g),
            'recall': recall_score(g['readmitted_30d'], g['pred'], zero_division=0),
            'precision': precision_score(g['readmitted_30d'], g['pred'], zero_division=0),
        })
    )
    print(summary)
    print()

--- race ---
                       n    recall  precision
race                                         
AfricanAmerican  18772.0  0.621685   0.189935
Asian              628.0  0.676923   0.257310
Caucasian        74220.0  0.636746   0.192911
Hispanic          2017.0  0.650943   0.219048
Other             1472.0  0.534722   0.190123

--- gender ---
                       n    recall  precision
gender                                       
Female           53454.0  0.648172   0.190595
Male             45886.0  0.609526   0.196433
Unknown/Invalid      3.0  0.000000   0.000000

--- age ---
                n    recall  precision
age                                   
[0-10)      160.0  0.000000   0.000000
[10-20)     690.0  0.300000   0.214286
[20-30)    1649.0  0.775424   0.322183
[30-40)    3764.0  0.615566   0.235772
[40-50)    9607.0  0.616211   0.218642
[50-60)   17060.0  0.568086   0.208866
[60-70)   22059.0  0.602487   0.196545
[70-80)   25331.0  0.647675   0.187506
[80-90)   16434.

## Notes
_(document any disparities found and how you'd mitigate them — e.g. threshold adjustment per subgroup, additional features, or flagging for human review)_